# Fluency-SpAM: analyses

1. **Semantic and phonological similarity** in the VFT and SpAM data (`phonetic_sim.py`).
2. **English vs Hindi word embeddings** for 32 animals (`temp.py`).

Both are ported from `Fluency SpAM Analysis/`. Run `../Data/data_cleaning.ipynb` first: section 1 reads `../Data/transliterated_data.pkl`.

**Requirements**

* `cc.hi.300.bin`, the Hindi fastText model (~7 GB), in this folder. If it is missing, `fasttext.util` downloads it here. It is gitignored.
* Section 2 downloads `word2vec-google-news-300` (~1.6 GB) through gensim into `~/gensim-data` the first time it runs. With both models loaded it needs roughly 11 GB of RAM.
* graph-tool (section 2) cannot be installed with pip. The project venv uses the system package, which it can see because the venv was created with `--system-site-packages`.

In [ ]:
import os
import fasttext as ft
import fasttext.util as util
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import epitran as epi
from sklearn.cluster import DBSCAN
from typing import Any, Optional

## Configuration

`SPAM_EXCLUSIONS` lists (participant ID, category) pairs to leave out of the cluster analysis, e.g. `{("P001", "Animal")}`.

In [ ]:
DATA_FILE = "../Data/transliterated_data.pkl"
MODEL_LANG = "hi"
MODEL_FILE = "cc.hi.300.bin"
EPITRAN_LANG = "hin-Deva"

SEM_COLOR = "#79bdf6"
PHO_COLOR = "#ffae3f"

CATEGORY_KEYS_VFT = {
    "Animal": "Animal VFT Words with Response Times",
    "Body Parts": "Body Parts VFT Words with Response Times",
    "Fruits and Vegetables": "Fruits and Vegetables VFT Words with Response Times",
}

CATEGORY_KEYS_SPAM = {
    "Animal":     "Animal SpAM Words with Normalised Coords",
    "Body Parts": "Body Parts SpAM Words with Normalised Coords",
    "Fruits and Vegetables": "Fruits and Vegetables SpAM Words with Normalised Coords",
}

SPAM_EXCLUSIONS = set()

## 1. Semantic and phonological similarity (`phonetic_sim.py`)

### Helpers

In [ ]:
def load_fasttext_model() -> Any:
    """Loads the FastText model for the configured language, downloading it first if the model file is not found on disk."""
    if not os.path.exists(MODEL_FILE):
        util.download_model(MODEL_LANG, if_exists="ignore")
    return ft.load_model(MODEL_FILE)


def levenshtein_distance(a: str, b: str) -> int:
    """Computes the character level Levenshtein edit distance between two strings."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]


def normalized_edit_similarity(a: str, b: str) -> float:
    """Returns the normalised edit similarity between two strings as a float in the range 0 to 1, where 1 indicates identical strings."""
    max_len = max(len(a), len(b), 1)
    return 1.0 - (levenshtein_distance(a, b) / max_len)


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Computes the cosine similarity between two vectors, returning 0 if either vector has zero magnitude."""
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)


def get_embedding(word: str, model: Any, cache: dict) -> np.ndarray:
    """Returns the FastText embedding for the given word, retrieving it from the cache if available and storing it otherwise.
    For multi-token words the embeddings of each token are averaged."""
    key = word.strip()
    if key in cache:
        return cache[key]

    tokens = key.split()
    if not tokens:
        vec = np.zeros(model.get_dimension(), dtype=np.float32)
    else:
        vecs = [model.get_word_vector(tok) for tok in tokens]
        vec = np.mean(vecs, axis=0)

    cache[key] = vec
    return vec


def get_ipa(word: str, epi_obj: Any, cache: dict) -> str:
    """Returns the IPA transliteration of the given word using the provided Epitran object, caching the result for future calls."""
    key = word.strip()
    if key in cache:
        return cache[key]
    ipa = epi_obj.transliterate(key)
    cache[key] = ipa
    return ipa


def parse_time_seconds(value: Any) -> Optional[float]:
    """Converts a millisecond value to seconds as a float, returning None if the value cannot be converted."""
    try:
        return float(value) / 1000.0
    except (TypeError, ValueError):
        return None

### Similarity along the VFT sequence

One change from `phonetic_sim.py`: in the per-participant plot, "Number of Words Retrieved" is now the number of words in the list. The script used the number of consecutive word pairs, which is one fewer.

In [ ]:
def collect_similarities(data: list, model: Any, epi_obj: Any) -> tuple:
    """Collects pairwise semantic and phonological similarity values for all consecutive word pairs across all participants and VFT categories.
    Each entry in the returned dictionaries is a tuple of cumulative retrieval time in seconds and the corresponding similarity value."""
    semantic = {cat: [] for cat in CATEGORY_KEYS_VFT}
    phonological = {cat: [] for cat in CATEGORY_KEYS_VFT}

    emb_cache = {}
    ipa_cache = {}

    for row in data:
        for cat, key in CATEGORY_KEYS_VFT.items():
            entries = row.get(key)
            if not isinstance(entries, list):
                continue

            pairs = [
                item
                for item in entries
                if isinstance(item, (list, tuple)) and len(item) >= 2
            ]
            if len(pairs) < 2:
                continue

            cumulative_times = []
            cum = 0.0
            for word, time_val in pairs:
                time_sec = parse_time_seconds(time_val)
                if time_sec is not None:
                    cum += time_sec
                cumulative_times.append(cum)

            for i in range(1, len(pairs)):
                prev_word = pairs[i - 1][0]
                curr_word = pairs[i][0]
                time_seconds = parse_time_seconds(pairs[i][1])
                if time_seconds is None:
                    continue

                prev_vec = get_embedding(prev_word, model, emb_cache)
                curr_vec = get_embedding(curr_word, model, emb_cache)
                sem_sim = cosine_similarity(prev_vec, curr_vec)

                prev_ipa = get_ipa(prev_word, epi_obj, ipa_cache)
                curr_ipa = get_ipa(curr_word, epi_obj, ipa_cache)
                pho_sim = normalized_edit_similarity(prev_ipa, curr_ipa)

                semantic[cat].append((cumulative_times[i], sem_sim))
                phonological[cat].append((cumulative_times[i], pho_sim))

    return semantic, phonological


def collect_avg_sim_per_participant(data: list, model: Any, epi_obj: Any) -> tuple:
    """Computes per-participant average semantic and phonological similarity for each VFT category.
    Each entry in the returned dictionaries is a tuple of the participant's average similarity and the number of words they retrieved."""
    semantic = {cat: [] for cat in CATEGORY_KEYS_VFT}
    phonological = {cat: [] for cat in CATEGORY_KEYS_VFT}

    emb_cache = {}
    ipa_cache = {}

    for row in data:
        for cat, key in CATEGORY_KEYS_VFT.items():
            avg_sem_sim = 0.0
            avg_pho_sim = 0.0
            count = 0

            entries = row.get(key)
            if not isinstance(entries, list):
                continue

            pairs = [
                item
                for item in entries
                if isinstance(item, (list, tuple)) and len(item) >= 2
            ]
            if len(pairs) < 2:
                continue

            for i in range(1, len(pairs)):
                prev_word = pairs[i - 1][0]
                curr_word = pairs[i][0]

                prev_vec = get_embedding(prev_word, model, emb_cache)
                curr_vec = get_embedding(curr_word, model, emb_cache)
                sem_sim = cosine_similarity(prev_vec, curr_vec)

                prev_ipa = get_ipa(prev_word, epi_obj, ipa_cache)
                curr_ipa = get_ipa(curr_word, epi_obj, ipa_cache)
                pho_sim = normalized_edit_similarity(prev_ipa, curr_ipa)

                avg_sem_sim = (count * avg_sem_sim + sem_sim) / (count + 1)
                avg_pho_sim = (count * avg_pho_sim + pho_sim) / (count + 1)
                count += 1
            # len(pairs) words; count is the number of consecutive pairs, one fewer
            semantic[cat].append((avg_sem_sim, len(pairs)))
            phonological[cat].append((avg_pho_sim, len(pairs)))

    return semantic, phonological


def add_trendline(ax: Any, xs: Any, ys: Any, color: str, label: str) -> None:
    """Fits a linear trendline to xs and ys and draws it on the given axes using the specified color and legend label."""
    if len(xs) < 2 or len(set(xs)) < 2:
        return

    coeffs = np.polyfit(xs, ys, 1)
    x_min, x_max = min(xs), max(xs)
    x_line = np.array([x_min, x_max], dtype=float)
    y_line = (coeffs[0] * x_line) + coeffs[1]
    ax.plot(x_line, y_line, color=color, linestyle="-", linewidth=2, label=label)


def plot_similarity_per_class(semantic_points: dict, phonological_points: dict) -> None:
    """Plots scatter graphs of pairwise semantic and phonological similarity against cumulative retrieval time for each VFT category, with a linear trendline for each."""
    fig, ax = plt.subplots(ncols=len(CATEGORY_KEYS_VFT) // 2 + 1, nrows=2, figsize=(18, 10))
    ax_flat = ax.flatten()

    for i in range(len(CATEGORY_KEYS_VFT)):
        cat = list(CATEGORY_KEYS_VFT.keys())[i]
        sem_points = semantic_points.get(cat, [])
        pho_points = phonological_points.get(cat, [])
        if not sem_points and not pho_points:
            continue

        if sem_points:
            xs, ys = zip(*sem_points)
            ax_flat[i].scatter(
                xs, ys, color=SEM_COLOR, alpha=0.1, s=18, label="Semantic"
            )
            add_trendline(ax_flat[i], xs, ys, SEM_COLOR, "Semantic Trend")

        if pho_points:
            xs, ys = zip(*pho_points)
            ax_flat[i].scatter(
                xs, ys, color=PHO_COLOR, alpha=0.1, s=18, label="Phonological"
            )
            add_trendline(ax_flat[i], xs, ys, PHO_COLOR, "Phonological Trend")

        ax_flat[i].set_title(f"{cat}: Similarity by Retrieval Time")
        ax_flat[i].set_xlabel("Retrieval Time (s)")
        ax_flat[i].set_ylabel("Similarity")
        ax_flat[i].grid(True, alpha=0.3)
        ax_flat[i].legend()

    for j in range(len(CATEGORY_KEYS_VFT), len(ax_flat)):
        ax_flat[j].set_visible(False)

    plt.tight_layout()
    plt.show()


def plot_avg_sim_per_participant(semantic_points: dict, phonological_points: dict) -> None:
    """Plots scatter graphs of each participant's average semantic and phonological similarity against their word count for each VFT category, with a linear trendline for each."""
    fig, ax = plt.subplots(ncols=len(CATEGORY_KEYS_VFT) // 2 + 1, nrows=2, figsize=(18, 10))
    ax_flat = ax.flatten()

    for i in range(len(CATEGORY_KEYS_VFT)):
        cat = list(CATEGORY_KEYS_VFT.keys())[i]
        sem_points = semantic_points.get(cat, [])
        pho_points = phonological_points.get(cat, [])
        if not sem_points and not pho_points:
            continue

        if sem_points:
            xs, ys = zip(*sem_points)
            ax_flat[i].scatter(
                xs, ys, color=SEM_COLOR, alpha=0.1, s=18
            )
            add_trendline(ax_flat[i], xs, ys, SEM_COLOR, "Semantic Similarity")

        if pho_points:
            xs, ys = zip(*pho_points)
            ax_flat[i].scatter(
                xs, ys, color=PHO_COLOR, alpha=0.1, s=18,
            )
            add_trendline(ax_flat[i], xs, ys, PHO_COLOR, "Phonological Similarity")

        ax_flat[i].set_title(f"{cat}: Similarity vs Number of Words Retrieved")
        ax_flat[i].set_xlabel("Avg Similarity")
        ax_flat[i].set_ylabel("Number of Words Retrieved")
        ax_flat[i].grid(True, alpha=0.3)
        ax_flat[i].legend()

    for j in range(len(CATEGORY_KEYS_VFT), len(ax_flat)):
        ax_flat[j].set_visible(False)

    plt.tight_layout()
    plt.show()

### Cluster vs switch similarity in SpAM

Two changes from `phonetic_sim.py`. The name check is replaced by `SPAM_EXCLUSIONS`. Empty or missing SpAM lists are skipped: a participant who typed no words in a category has no transitions to count, and `DBSCAN` cannot fit zero points.

In [ ]:
def part_designated_clusters(data: list, cat: str, name: str) -> list:
    """Returns per-transition cluster labels for the named participant in the given SpAM category using DBSCAN on normalised coordinates.
    Labels are 0 for a within-cluster transition, 1 for a switch, and 2 for unlabellable."""
    key = CATEGORY_KEYS_SPAM[cat]
    person = None
    for person in data:
        if person["Name"] == name:
            break
    coords = []
    if (person["Name"], cat) not in SPAM_EXCLUSIONS:
        for x in range(len(person[key])):
            coords.append((person[key][x][1], person[key][x][2]))
        points = np.array(coords)
        db = DBSCAN(eps=0.11, min_samples=1).fit(points)
        part_clusters = []
        for _ in range(len(points)):
            part_clusters.append(2)
        for i in range(len(db.labels_) - 1):
            if db.labels_[i] == db.labels_[i + 1]:
                part_clusters[i] = 0 # cluster
            else:
                part_clusters[i] = 1 # switch (last element in the cluster)
    return part_clusters


def sim_drop_clusters(data: list, cat: str, model: Any, name: str) -> list:
    """Returns per-word cluster labels for the named participant in the given SpAM category using the similarity drop method.
    A switch is marked at position i+1 when the similarity from word i to word i+1 is a local minimum relative to its neighbours.
    Labels are 0 for cluster, 1 for switch, and 2 for unlabellable."""
    key = CATEGORY_KEYS_SPAM[cat]
    person = None
    for person in data:
        if person["Name"] == name:
            break
    word_list = []
    for j in range(len(person[key])):
        word_list.append(person[key][j][0])
    emb_cache = {}
    label = [2] * len(word_list) # 2 for unlabellable
    if len(word_list) <= 3:
        return label
    for i in range(0, len(person[key]), 1):
        if i + 3 >= len(person[key]):
            break
        w1, w2, w3, w4 = person[key][i][0], person[key][i+1][0], person[key][i+2][0], person[key][i+3][0]
        v1 = get_embedding(w1, model, emb_cache)
        emb_cache[w1.strip()] = v1
        v2 = get_embedding(w2, model, emb_cache)
        emb_cache[w2.strip()] = v2
        v3 = get_embedding(w3, model, emb_cache)
        emb_cache[w3.strip()] = v3
        v4 = get_embedding(w4, model, emb_cache)
        emb_cache[w4.strip()] = v4
        sem_sim_12 = cosine_similarity(v1, v2)
        sem_sim_23 = cosine_similarity(v2, v3)
        sem_sim_34 = cosine_similarity(v3, v4)
        if sem_sim_12 > sem_sim_23 < sem_sim_34:
            label[i+1] = 1 # switch
        else:
            label[i+1] = 0 # cluster
    return label


def mean_plot_clusterwise_similarity(data: list, model: Any, epi_obj: Any) -> None:
    """Computes mean cluster and switch similarity across participants for each SpAM category using both the similarity drop and participant designated methods.
    Produces one figure per category containing two bar chart subplots, one for semantic similarity and one for phonological similarity."""
    for cat in CATEGORY_KEYS_SPAM:
        key = CATEGORY_KEYS_SPAM[cat]
        mean_cluster_sem_sim_drop, mean_cluster_pho_sim_drop = 0.0, 0.0
        mean_cluster_sem_part_des, mean_cluster_pho_part_des = 0.0, 0.0
        mean_switch_sem_sim_drop, mean_switch_pho_sim_drop = 0.0, 0.0
        mean_switch_sem_part_des, mean_switch_pho_part_des = 0.0, 0.0
        C_cluster_pd, C_switch_pd, C_cluster_sd, C_switch_sd = 0, 0, 0, 0
        for person in data:
            mean_cluster_sem_sim_drop_pp, mean_cluster_pho_sim_drop_pp = 0.0, 0.0
            mean_cluster_sem_part_des_pp, mean_cluster_pho_part_des_pp = 0.0, 0.0
            mean_switch_sem_sim_drop_pp, mean_switch_pho_sim_drop_pp = 0.0, 0.0
            mean_switch_sem_part_des_pp, mean_switch_pho_part_des_pp = 0.0, 0.0
            if (person["Name"], cat) in SPAM_EXCLUSIONS:
                continue
            if not isinstance(person.get(key), list) or not person[key]:
                continue
            part_clusters = part_designated_clusters(data, cat, person["Name"])
            simdrop_clusters = sim_drop_clusters(data, cat, model, person["Name"])
            word_list = []
            for j in range(len(person[key])):
                word_list.append(person[key][j][0])
            emb_cache = {}
            ipa_cache = {}
            c_cluster_pd, c_switch_pd, c_cluster_sd, c_switch_sd = 0, 0, 0, 0
            for i in range(len(word_list) - 1):
                w1, w2 = word_list[i], word_list[i+1]
                v1 = get_embedding(w1, model, emb_cache)
                emb_cache[w1.strip()] = v1
                v2 = get_embedding(w2, model, emb_cache)
                emb_cache[w2.strip()] = v2
                sem_sim = cosine_similarity(v1, v2)
                i1, i2 = get_ipa(w1, epi_obj, ipa_cache), get_ipa(w2, epi_obj, ipa_cache)
                pho_sim = normalized_edit_similarity(i1, i2)

                if part_clusters[i] == 1: # Switch case
                    mean_switch_sem_part_des_pp = (mean_switch_sem_part_des_pp * c_switch_pd + sem_sim) / (c_switch_pd + 1)
                    mean_switch_pho_part_des_pp = (mean_switch_pho_part_des_pp * c_switch_pd + pho_sim) / (c_switch_pd + 1)
                    c_switch_pd += 1
                elif part_clusters[i] == 0:
                    mean_cluster_sem_part_des_pp = (mean_cluster_sem_part_des_pp * c_cluster_pd + sem_sim) / (c_cluster_pd + 1)
                    mean_cluster_pho_part_des_pp = (mean_cluster_pho_part_des_pp * c_cluster_pd + pho_sim) / (c_cluster_pd + 1)
                    c_cluster_pd += 1

                if simdrop_clusters[i] == 2:
                    continue
                elif simdrop_clusters[i] == 1: # Switch case
                    mean_switch_sem_sim_drop_pp = (mean_switch_sem_sim_drop_pp * c_switch_sd + sem_sim) / (c_switch_sd + 1)
                    mean_switch_pho_sim_drop_pp = (mean_switch_pho_sim_drop_pp * c_switch_sd + pho_sim) / (c_switch_sd + 1)
                    c_switch_sd += 1
                elif simdrop_clusters[i] == 0:
                    mean_cluster_sem_sim_drop_pp = (mean_cluster_sem_sim_drop_pp * c_cluster_sd + sem_sim) / (c_cluster_sd + 1)
                    mean_cluster_pho_sim_drop_pp = (mean_cluster_pho_sim_drop_pp * c_cluster_sd + pho_sim) / (c_cluster_sd + 1)
                    c_cluster_sd += 1

            if c_cluster_pd > 0:
                mean_cluster_sem_part_des = (mean_cluster_sem_part_des * C_cluster_pd + mean_cluster_sem_part_des_pp) / (C_cluster_pd + 1)
                mean_cluster_pho_part_des = (mean_cluster_pho_part_des * C_cluster_pd + mean_cluster_pho_part_des_pp) / (C_cluster_pd + 1)
                C_cluster_pd += 1

            if c_switch_pd > 0:
                mean_switch_sem_part_des = (mean_switch_sem_part_des * C_switch_pd + mean_switch_sem_part_des_pp) / (C_switch_pd + 1)
                mean_switch_pho_part_des = (mean_switch_pho_part_des * C_switch_pd + mean_switch_pho_part_des_pp) / (C_switch_pd + 1)
                C_switch_pd += 1

            if c_cluster_sd > 0:
                mean_cluster_sem_sim_drop = (mean_cluster_sem_sim_drop * C_cluster_sd + mean_cluster_sem_sim_drop_pp) / (C_cluster_sd + 1)
                mean_cluster_pho_sim_drop = (mean_cluster_pho_sim_drop * C_cluster_sd + mean_cluster_pho_sim_drop_pp) / (C_cluster_sd + 1)
                C_cluster_sd += 1

            if c_switch_sd > 0:
                mean_switch_sem_sim_drop = (mean_switch_sem_sim_drop * C_switch_sd + mean_switch_sem_sim_drop_pp) / (C_switch_sd + 1)
                mean_switch_pho_sim_drop = (mean_switch_pho_sim_drop * C_switch_sd + mean_switch_pho_sim_drop_pp) / (C_switch_sd + 1)
                C_switch_sd += 1

        fig, ax = plt.subplots(nrows=2, ncols=1, sharex=False, sharey=False, figsize=(12, 8))
        bar_width = 0.25

        cluster_sem = [mean_cluster_sem_sim_drop, mean_cluster_sem_part_des]
        switch_sem  = [mean_switch_sem_sim_drop,  mean_switch_sem_part_des]
        cluster_pho = [mean_cluster_pho_sim_drop, mean_cluster_pho_part_des]
        switch_pho  = [mean_switch_pho_sim_drop,  mean_switch_pho_part_des]

        br1 = np.arange(2)                        # [0, 1]  one position per method
        br2 = [x + bar_width for x in br1]        # [0.25, 1.25]

        ax[0].bar(br1, cluster_sem, color=SEM_COLOR, width=bar_width, label="cluster")
        ax[0].bar(br2, switch_sem,  color=PHO_COLOR,  width=bar_width, label="switch")
        ax[0].set_ylabel("mean semantic similarity")
        ax[0].set_xticks([x + bar_width / 2 for x in br1])
        ax[0].set_xticklabels(["similarity drop", "participant designated"])
        ax[0].legend()

        ax[1].bar(br1, cluster_pho, color=SEM_COLOR, width=bar_width, label="cluster")
        ax[1].bar(br2, switch_pho,  color=PHO_COLOR,  width=bar_width, label="switch")
        ax[1].set_ylabel("mean phonological similarity")
        ax[1].set_xticks([x + bar_width / 2 for x in br1])
        ax[1].set_xticklabels(["similarity drop", "participant designated"])
        ax[1].legend()

        ax[1].set_xlabel("switch method")
        fig.suptitle(cat)
        plt.tight_layout()
        plt.show()

### Run

In [ ]:
model = load_fasttext_model()
epi_obj = epi.Epitran(EPITRAN_LANG)

In [ ]:
with open(DATA_FILE, "rb") as f:
    data = pkl.load(f)
print(f"{len(data)} participants")

In [ ]:
semantic, phonological = collect_similarities(data, model, epi_obj)

plot_similarity_per_class(semantic, phonological)

In [ ]:
semantic_avg, phonological_avg = collect_avg_sim_per_participant(data, model, epi_obj)
plot_avg_sim_per_participant(semantic_avg, phonological_avg)

In [ ]:
mean_plot_clusterwise_similarity(data, model, epi_obj)

## 2. English vs Hindi word embeddings (`temp.py`)

Pairwise similarities of 32 animals in English word2vec (Google News) and Hindi fastText, the Spearman correlation between the two, and word clouds of similarity to "dog" / "कुत्ता" (saved as `animal_similarity.png`). `temp.py` defined its own copies of `load_fasttext_model`, `cosine_similarity` and `get_embedding`, which are identical to those above, and loaded the fastText model a second time; here it reuses `model`.

In [ ]:
import gensim.downloader as api
from scipy.stats import spearmanr, rankdata, linregress
import graph_tool.all as gt

eng_model = api.load("word2vec-google-news-300")
hin_model = model  # the cc.hi.300.bin model loaded in section 1

In [ ]:
eng_words = ["Dog", "Cat", "Cow", "Buffalo", "Ox", "Goat", "Sheep", "Horse", "Donkey", "Rat", "Camel", "Elephant", "Monkey", "Lion", "Tiger", "Bear", "Fox", "Wolf", "Deer", "Rabbit", "Snake", "Frog", "Fish", "Turtle", "Bird", "Hen", "Pig", "Parrot", "Crow", "Peacock", "Rooster", "Pigeon"]
hin_words = ["कुत्ता", "बिल्ली", "गाय", "भैंस", "बैल", "बकरी", "भेड़", "घोड़ा", "गधा", "चूहा", "ऊँट", "हाथी", "बंदर", "शेर", "बाघ", "भालू", "लोमड़ी", "भेड़िया", "हिरण", "खरगोश", "साँप", "मेंढक", "मछली", "कछुआ", "चिड़िया", "मुर्गी", "सूअर", "तोता", "कौआ", "मोर", "मुर्गा", "कबूतर"]
n = len(eng_words)
eng_sim_matrix = np.zeros((n, n))
hin_sim_matrix = np.zeros((n, n))
i = 0
cache = {}

for k1, v1 in zip(eng_words, hin_words):
    j = 0
    for k2, v2 in zip(eng_words, hin_words):
        eng_sim_matrix[i][j] = eng_model.similarity(k1.lower(), k2.lower())
        e1, e2 = get_embedding(v1, hin_model, cache), get_embedding(v2, hin_model, cache)
        hin_sim_matrix[i][j] = cosine_similarity(e1, e2)
        j += 1
    i += 1

eng_upper_tri = eng_sim_matrix[np.triu_indices_from(eng_sim_matrix, k=1)]
hin_upper_tri = hin_sim_matrix[np.triu_indices_from(hin_sim_matrix, k=1)]
eng_ranks = rankdata(eng_upper_tri)
hin_ranks = rankdata(hin_upper_tri)
slope, intercept, *_ = linregress(eng_upper_tri, hin_upper_tri)
x_line = np.linspace(eng_upper_tri.min(), eng_upper_tri.max(), 200)

plt.figure(figsize=(8, 6))
plt.scatter(eng_upper_tri, hin_upper_tri, alpha=0.3)
plt.plot(x_line, slope * x_line + intercept, color="red")
plt.xlabel("English pairwise similarity")
plt.ylabel("Hindi pairwise similarity")
plt.show()

r, p = spearmanr(eng_upper_tri, hin_upper_tri)
print("Spearman rho = ", r, "p = ", p)

In [ ]:
def get_graph_positions(n_nodes: int, centre_idx: int, sim_matrix: np.ndarray) -> np.ndarray:
    g = gt.Graph(directed=False)
    g.add_vertex(n_nodes)
    
    e_weight = g.new_edge_property("double")
    for idx in range(n_nodes):
        if idx == centre_idx:
            continue
        e = g.add_edge(centre_idx, idx)
        e_weight[e] = sim_matrix[centre_idx][idx]

    pin = g.new_vertex_property("bool")
    pin[g.vertex(centre_idx)] = True

    pos_init = g.new_vertex_property("vector<double>")
    pos_init[g.vertex(centre_idx)] = [0.0, 0.0]

    angle_idx = 0
    for idx in range(n_nodes):
        if idx == centre_idx:
            continue
        angle = 2 * np.pi * angle_idx / (n_nodes - 1)
        pos_init[g.vertex(idx)] = [np.cos(angle), np.sin(angle)]
        angle_idx += 1

    pos = gt.sfdp_layout(
        g,
        pos=pos_init,
        pin=pin,
        eweight=e_weight,
        max_iter=500
    )
    return np.array([pos[g.vertex(idx)] for idx in range(n_nodes)])

positions_eng = get_graph_positions(n, 0, eng_sim_matrix)
positions_hin = get_graph_positions(n, 0, hin_sim_matrix)

def draw_similarity_panel(ax, positions, words, sim_matrix, centre_idx, centre_label, title, font_family):
    """
    Renders one word-cloud panel.
    """
    sim_to_centre = sim_matrix[centre_idx, :] 

    peripheral_sims = np.delete(sim_to_centre, centre_idx)
    sim_min, sim_max = peripheral_sims.min(), peripheral_sims.max()

    def normalise(s):
        if sim_max == sim_min:
            return 0.5
        return (s - sim_min) / (sim_max - sim_min)

    for idx, word in enumerate(words):
        if idx == centre_idx:
            continue

        x, y = positions[idx]
        norm = normalise(sim_to_centre[idx])

        # Increased base values for better visibility of distant words
        fontsize  = 9 + 11 * norm 
        alpha     = 1
        colour    = plt.cm.cool(norm) 

        ax.plot([0, x], [0, y], color="lightgrey", linewidth=0.6, zorder=0)

        ax.text(x, y, word,
                fontsize=fontsize, alpha=alpha, color=colour,
                ha="center", va="center", fontfamily=font_family,
                fontweight="bold" if norm > 0.65 else "normal",
                zorder=2)

    ax.text(0, 0, centre_label,
            fontsize=20, fontweight="bold",
            ha="center", va="center",
            color="white", zorder=3, fontfamily=font_family,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="crimson",
                edgecolor="darkred",
                linewidth=2
            ))

    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=15, pad=12)

fig, (ax_eng, ax_hin) = plt.subplots(1, 2, figsize=(16, 8))

draw_similarity_panel(ax_eng, positions_eng, eng_words, eng_sim_matrix,
                      centre_idx=0, centre_label="Dog",
                      title="English — similarity to Dog", font_family="sans-serif")

draw_similarity_panel(ax_hin, positions_hin, hin_words, hin_sim_matrix,
                      centre_idx=0, centre_label="कुत्ता",
                      title="Hindi — similarity to कुत्ता", font_family="Noto Sans Devanagari")

plt.tight_layout()
plt.savefig("animal_similarity.png", dpi=150, bbox_inches="tight")
plt.show()